# Historical Submission Scoring

Computes **Brier scores** for all archived submission CSVs against actual tournament results.

**Brier score** = mean((pred - actual)²) over all played games.  
Lower is better. Random guessing = 0.25. Perfect predictions = 0.0.

Scorable years: **2022, 2023, 2024, 2025** (results in `data/2026/`).

Archive structure:
```
archive/submissions/{year}/{model}.csv
```

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.insert(0, os.path.abspath('..'))

from src.evaluation.scorer import (
    load_tournament_results,
    score_submission,
    score_all_submissions,
    generate_scoring_report,
)

ARCHIVE_DIR = "../archive/submissions"
DATA_DIR = "../data/2026"   # cumulative results through 2025
SEASONS = [2022, 2023, 2024, 2025]

print("Evaluation setup ready.")

## Score All Submissions

In [ ]:
results_df = score_all_submissions(
    archive_dir=ARCHIVE_DIR,
    data_dir=DATA_DIR,
    seasons=SEASONS,
)
print(f"Scored {len(results_df)} model×year×gender combinations.")
results_df

## Summary Report

In [ ]:
report = generate_scoring_report(results_df)
print(report)

## Visualization — Brier Score by Model × Year

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=False)

for ax, gender in zip(axes, ['M', 'W']):
    gdf = results_df[results_df['gender'] == gender]
    if gdf.empty:
        ax.set_visible(False)
        continue

    pivot = gdf.pivot_table(index='model', columns='year', values='brier_score', aggfunc='first')
    pivot = pivot.sort_values(pivot.columns.tolist(), na_position='last')

    x = np.arange(len(pivot))
    width = 0.8 / len(pivot.columns)
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(pivot.columns)))

    for i, (year, color) in enumerate(zip(pivot.columns, colors)):
        vals = pivot[year].values
        bars = ax.bar(x + i * width, vals, width, label=str(year), color=color, alpha=0.85)
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                        f'{val:.3f}', ha='center', va='bottom', fontsize=7)

    # Random baseline
    ax.axhline(y=0.25, color='red', linestyle='--', alpha=0.6, label='Random (0.25)')

    gender_label = "Men's" if gender == 'M' else "Women's"
    ax.set_title(f"{gender_label} Tournament Brier Scores", fontsize=13, fontweight='bold')
    ax.set_xlabel("Model", fontsize=11)
    ax.set_ylabel("Brier Score (lower = better)", fontsize=11)
    ax.set_xticks(x + width * (len(pivot.columns) - 1) / 2)
    ax.set_xticklabels(pivot.index, rotation=30, ha='right', fontsize=9)
    ax.legend(fontsize=9)
    ax.set_ylim(0, 0.28)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.grid(axis='y', alpha=0.3)

plt.suptitle("Historical March Madness Submission Performance", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../output/historical_brier_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## Round-by-Round Breakdown

In [ ]:
# Detailed round-by-round scoring for best performing models
print("Round-by-round Brier scores for men's submissions...\n")

from pathlib import Path

round_rows = []
for season in SEASONS:
    year_dir = Path(ARCHIVE_DIR) / str(season)
    if not year_dir.exists(): continue

    try:
        results_m = load_tournament_results(DATA_DIR, season, gender='M')
    except FileNotFoundError:
        continue

    for csv_file in sorted(year_dir.glob('*.csv')):
        try:
            sub = pd.read_csv(csv_file)
        except Exception:
            continue
        if 'ID' not in sub.columns or 'Pred' not in sub.columns: continue

        stats = score_submission(sub, results_m, season, 'M')
        if stats['n_games'] == 0: continue

        for round_name, brier in stats['per_round'].items():
            round_rows.append({
                'year': season, 'model': csv_file.stem,
                'round': round_name, 'brier': brier,
            })

round_df = pd.DataFrame(round_rows)
if not round_df.empty:
    pivot_r = round_df.pivot_table(index=['model','year'], columns='round', values='brier', aggfunc='first')
    # Order rounds logically
    round_order = ['Round of 64','Round of 32','Sweet 16','Elite 8','Final Four','Championship']
    pivot_r = pivot_r[[c for c in round_order if c in pivot_r.columns]]
    display(pivot_r.round(4))
else:
    print("No round-by-round data available.")

## Calibration Analysis

In [ ]:
# Reliability diagram: are predicted 70% probabilities winning 70% of games?
print("Calibration analysis for best models...")

best_models_per_year = results_df[results_df['gender']=='M'].sort_values('brier_score').groupby('year').first().reset_index()

fig, axes = plt.subplots(1, len(SEASONS), figsize=(16, 5), sharey=True)

for ax, season in zip(axes, SEASONS):
    best_row = best_models_per_year[best_models_per_year['year'] == season]
    if len(best_row) == 0:
        ax.set_visible(False)
        continue

    model_name = best_row.iloc[0]['model']
    csv_path = Path(ARCHIVE_DIR) / str(season) / f"{model_name}.csv"
    try:
        sub = pd.read_csv(csv_path)
        results_m = load_tournament_results(DATA_DIR, season, 'M')
    except FileNotFoundError:
        ax.set_visible(False)
        continue

    pred_lookup = dict(zip(sub['ID'], sub['Pred']))
    preds, actuals = [], []
    for _, g in results_m.iterrows():
        w, l = int(g['WTeamID']), int(g['LTeamID'])
        lo, hi = min(w,l), max(w,l)
        gid = f"{season}_{lo}_{hi}"
        if gid in pred_lookup:
            preds.append(pred_lookup[gid])
            actuals.append(1.0 if lo == w else 0.0)

    if not preds: continue

    preds_arr = np.array(preds)
    actuals_arr = np.array(actuals)

    # Bin predictions
    bins = np.linspace(0, 1, 11)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_actuals = []
    for lo_b, hi_b in zip(bins[:-1], bins[1:]):
        mask = (preds_arr >= lo_b) & (preds_arr < hi_b)
        bin_actuals.append(actuals_arr[mask].mean() if mask.sum() > 0 else np.nan)

    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    ax.plot(bin_centers, bin_actuals, 'o-', color='steelblue', markersize=6)
    ax.set_title(f"{season}\n{model_name}", fontsize=10)
    ax.set_xlabel('Predicted Prob')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Actual Win Rate')
plt.suptitle("Calibration — Best Model Per Year (Men's)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../output/calibration_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

## Upset Analysis

In [ ]:
# How well do models call upsets (lower-seeded team beating higher-seeded)?
print("Upset detection analysis...")

upset_rows = []
for season in SEASONS:
    year_dir = Path(ARCHIVE_DIR) / str(season)
    if not year_dir.exists(): continue

    try:
        results_m = load_tournament_results(DATA_DIR, season, 'M')
        seeds_df = pd.read_csv(f"{DATA_DIR}/MNCAATourneySeeds.csv")
        seeds_season = seeds_df[seeds_df['Season'] == season]
        seed_map = {}
        for _, row in seeds_season.iterrows():
            seed_map[row['TeamID']] = int(row['Seed'][1:3])
    except Exception:
        continue

    # Find upsets (lower seed won = higher seed number won)
    upsets = []
    for _, g in results_m.iterrows():
        w_seed = seed_map.get(g['WTeamID'], 8)
        l_seed = seed_map.get(g['LTeamID'], 8)
        if w_seed > l_seed:  # upset: higher-number seed (weaker) beat lower-number (stronger)
            upsets.append({'WTeamID': g['WTeamID'], 'LTeamID': g['LTeamID'],
                          'WSeed': w_seed, 'LSeed': l_seed})

    if not upsets: continue

    for csv_file in sorted(year_dir.glob('*.csv')):
        try:
            sub = pd.read_csv(csv_file)
        except Exception: continue
        if 'ID' not in sub.columns or 'Pred' not in sub.columns: continue
        pred_lookup = dict(zip(sub['ID'], sub['Pred']))

        correct_upsets = 0
        for u in upsets:
            lo, hi = min(u['WTeamID'], u['LTeamID']), max(u['WTeamID'], u['LTeamID'])
            gid = f"{season}_{lo}_{hi}"
            if gid in pred_lookup:
                pred = pred_lookup[gid]
                # Upset winner is WTeamID; predicted upset if P(lower_id wins) > 0.5
                predicted_winner = lo if pred > 0.5 else hi
                if predicted_winner == u['WTeamID']:
                    correct_upsets += 1

        upset_rows.append({
            'year': season, 'model': csv_file.stem,
            'n_upsets': len(upsets),
            'correct_upsets': correct_upsets,
            'upset_acc': correct_upsets / len(upsets) if upsets else np.nan,
        })

upset_df = pd.DataFrame(upset_rows)
if not upset_df.empty:
    pivot_u = upset_df.pivot_table(index='model', columns='year', values='upset_acc', aggfunc='first')
    print("Upset detection accuracy (fraction of actual upsets correctly predicted):")
    display(pivot_u.round(3))
else:
    print("No upset data available.")

## Key Insights

Based on the Brier score analysis, the following insights emerge:

### What to look for:
- **Best overall model**: Lowest average Brier score across years
- **Consistency**: Models with low variance across years are more reliable
- **Early round performance**: Round of 64 should have lowest Brier (more predictable upsets)
- **Calibration gaps**: If the reliability diagram shows systematic over/under-confidence, add a calibration layer
- **Upset detection**: Models rarely predict upsets — if upset_acc < 20%, the model is too conservative

### Potential improvements to try:
1. **Calibration layer**: Apply isotonic regression post-prediction (`sklearn.calibration.CalibratedClassifierCV`)
2. **Margin of victory features**: Add avg point differential to `MLModel.py`
3. **Coach tenure**: Use `MTeamCoaches.csv` — coaches in their first year underperform
4. **Massey trajectory**: Add ranking improvement trend from mid-season to tournament
5. **Women's SOS**: Construct strength-of-schedule proxy for women's (no Massey available)
6. **Bradley-Terry model**: Fit pairwise strength parameters across full season (often outperforms ELO)